# Notebook 3 — Classification & Evaluation
**Project:** Chest X-ray Pneumonia Detection — Generalization Study

This notebook:
1. Trains 4 classifiers on K-Means features: Logistic Regression, Random Forest, SVM, XGBoost
2. Repeats on raw PCA features (baseline ablation)
3. Tunes XGBoost with RandomizedSearchCV
4. Evaluates all models: accuracy, precision, recall, F1, ROC-AUC
5. Saves results and best model for Notebook 4

In [ ]:
!pip install scikit-learn xgboost numpy matplotlib seaborn --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve)
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

## Step 1 — Load features

In [ ]:
# K-Means features
X_train_feat = np.load('preprocessed/X_train_feat.npy')
X_test_feat  = np.load('preprocessed/X_test_feat.npy')
y_train      = np.load('preprocessed/y_train.npy')
y_test       = np.load('preprocessed/y_test.npy')

# Raw baseline features
X_train_raw  = np.load('preprocessed/X_train_raw.npy')
X_test_raw   = np.load('preprocessed/X_test_raw.npy')

# Scale K-Means features
scaler = StandardScaler()
X_train_feat_s = scaler.fit_transform(X_train_feat)
X_test_feat_s  = scaler.transform(X_test_feat)

print(f'K-Means features: train={X_train_feat_s.shape}, test={X_test_feat_s.shape}')
print(f'Raw features    : train={X_train_raw.shape}, test={X_test_raw.shape}')

## Step 2 — Define evaluation helper

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_prob) if y_prob is not None else 0.0
    
    print(f'{model_name:30s}  Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
    return {'model': model_name, 'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1, 'roc_auc': auc,
            'y_pred': y_pred, 'y_prob': y_prob}

## Step 3 — Train & evaluate all 4 models on K-Means features

In [ ]:
print('=== K-Means Features ===')
results_kmeans = []

# 1. Logistic Regression (baseline)
lr = LogisticRegression(random_state=SEED, max_iter=1000, class_weight='balanced')
lr.fit(X_train_feat_s, y_train)
results_kmeans.append(evaluate_model(lr, X_test_feat_s, y_test, 'Logistic Regression'))

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, class_weight='balanced')
rf.fit(X_train_feat_s, y_train)
results_kmeans.append(evaluate_model(rf, X_test_feat_s, y_test, 'Random Forest'))

# 3. SVM
svm = SVC(kernel='rbf', probability=True, random_state=SEED, class_weight='balanced')
svm.fit(X_train_feat_s, y_train)
results_kmeans.append(evaluate_model(svm, X_test_feat_s, y_test, 'SVM (RBF)'))

# 4. XGBoost (default first)
xgb = XGBClassifier(random_state=SEED, use_label_encoder=False,
                     eval_metric='logloss', scale_pos_weight=2)
xgb.fit(X_train_feat_s, y_train)
results_kmeans.append(evaluate_model(xgb, X_test_feat_s, y_test, 'XGBoost (default)'))

## Step 4 — Tune XGBoost with RandomizedSearchCV

In [ ]:
print('Tuning XGBoost with RandomizedSearchCV (this takes ~3-5 minutes)...')

param_dist = {
    'n_estimators':    [100, 200, 300, 400],
    'max_depth':       [3, 4, 5, 6, 7],
    'learning_rate':   [0.01, 0.05, 0.1, 0.2],
    'subsample':       [0.6, 0.8, 1.0],
    'colsample_bytree':[0.6, 0.8, 1.0],
    'min_child_weight':[1, 3, 5]
}

xgb_base = XGBClassifier(random_state=SEED, use_label_encoder=False,
                          eval_metric='logloss', scale_pos_weight=2)

search = RandomizedSearchCV(
    xgb_base, param_dist,
    n_iter=20, cv=3, scoring='roc_auc',
    random_state=SEED, n_jobs=-1, verbose=1
)
search.fit(X_train_feat_s, y_train)

print(f'\nBest params: {search.best_params_}')
print(f'Best CV AUC: {search.best_score_:.4f}')

xgb_tuned = search.best_estimator_
results_kmeans.append(evaluate_model(xgb_tuned, X_test_feat_s, y_test, 'XGBoost (tuned)'))

## Step 5 — Baseline ablation: same models on raw PCA features

In [ ]:
print('\n=== Raw PCA Features (Baseline - No K-Means) ===')
results_raw = []

lr_raw = LogisticRegression(random_state=SEED, max_iter=1000, class_weight='balanced')
lr_raw.fit(X_train_raw, y_train)
results_raw.append(evaluate_model(lr_raw, X_test_raw, y_test, 'LR (raw pixels)'))

rf_raw = RandomForestClassifier(n_estimators=200, random_state=SEED, class_weight='balanced')
rf_raw.fit(X_train_raw, y_train)
results_raw.append(evaluate_model(rf_raw, X_test_raw, y_test, 'RF (raw pixels)'))

xgb_raw = XGBClassifier(random_state=SEED, use_label_encoder=False,
                         eval_metric='logloss', scale_pos_weight=2)
xgb_raw.fit(X_train_raw, y_train)
results_raw.append(evaluate_model(xgb_raw, X_test_raw, y_test, 'XGB (raw pixels)'))

## Step 6 — Confusion matrices for all K-Means models

In [ ]:
model_names = ['Logistic\nRegression', 'Random\nForest', 'SVM', 'XGBoost\n(default)', 'XGBoost\n(tuned)']
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Confusion Matrices — K-Means Features', fontsize=13)

for i, (res, name) in enumerate(zip(results_kmeans, model_names)):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'])
    axes[i].set_title(name, fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True' if i == 0 else '')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as confusion_matrices.png')

## Step 7 — ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for res, color in zip(results_kmeans, colors):
    if res['y_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f"{res['model']} (AUC={res['roc_auc']:.3f})")

ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — K-Means Features')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as roc_curves.png')

## Step 8 — Feature importance from Random Forest and XGBoost

In [ ]:
feature_names = [
    'Prop_cluster0', 'Prop_cluster1', 'Prop_cluster2',
    'Mean_cluster0', 'Mean_cluster1', 'Mean_cluster2',
    'Std_cluster0',  'Std_cluster1',  'Std_cluster2'
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest importance
rf_imp = rf.feature_importances_
axes[0].barh(feature_names, rf_imp, color='#4CAF50')
axes[0].set_title('Random Forest Feature Importance')
axes[0].set_xlabel('Importance')

# XGBoost importance
xgb_imp = xgb_tuned.feature_importances_
axes[1].barh(feature_names, xgb_imp, color='#2196F3')
axes[1].set_title('XGBoost (tuned) Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as feature_importance.png')

## Step 9 — Save best model and results

In [ ]:
import pickle, os
os.makedirs('models', exist_ok=True)

# Save best model (tuned XGBoost) and scaler
with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(xgb_tuned, f)
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save all results for report
import json
results_to_save = []
for r in results_kmeans + results_raw:
    results_to_save.append({
        'model': r['model'], 'accuracy': float(r['accuracy']),
        'precision': float(r['precision']), 'recall': float(r['recall']),
        'f1': float(r['f1']), 'roc_auc': float(r['roc_auc'])
    })

with open('models/results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)

print('Best model saved to models/best_model.pkl')
print('All results saved to models/results.json')
print('Notebook 3 complete. Open notebook 4 (generalization) next.')